## Data Processing

----

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as pex
from datetime import datetime,time

In [5]:
def canonize_id(id: str):
    id = id.replace('-','')
    
    while (len(id)<8):
        id = '0' + id
    
    if len(id) == 9:
        id = id[:-1]
    return id

### 220810

#### Load Raw Data

CXL

In [ ]:
df_cxl = pd.read_excel('raw_data/CXL_data.xlsx')

In [ ]:
df_cxl = df_cxl.rename(columns={'ת':'id','מנ':'current_disease'})

In [ ]:
df_cxl.id = df_cxl.id.astype(str).apply(canonize_id)

In [ ]:
df_cxl['eye_id']='N'

df_cxl.loc[(df_cxl.current_disease.str.contains('שמאל')) & (df_cxl.current_disease.str.contains('ימין') == False),'eye_id'] = 'L'
df_cxl.loc[(df_cxl.current_disease.str.contains('שמאל') == False) & (df_cxl.current_disease.str.contains('ימין')),'eye_id'] = 'R'
df_cxl.loc[(df_cxl.current_disease.str.contains('שמאל')) & (df_cxl.current_disease.str.contains('ימין')),'eye_id'] = 'B'
df_cxl.loc[(df_cxl.current_disease.str.contains('שמאל') == False) & (df_cxl.current_disease.str.contains('ימין') == False),'eye_id'] = 'N'

df_cxl.loc[~df_cxl['first KERATOCONUS dg name right side'].isna(),'eye_id'] = 'R'
df_cxl.loc[~df_cxl['first KERATOCONUS dg name left side'].isna(),'eye_id'] = 'L'
df_cxl.loc[~df_cxl['first KERATOCONUS dg name both side'].isna(),'eye_id'] = 'BC'

In [ ]:
pex.histogram(df_cxl,x='eye_id',title = 'Which Eye Diagnostics')

Pentacam

In [ ]:
df_penta = pd.read_excel('raw_data/Pentacam_raw_data.xlsx')

In [ ]:
df_penta = df_penta.rename(columns={'Pat-ID:':'id'})
df_penta.id = df_penta.id.astype(str).apply(canonize_id)

#### IDs List

Pentacam data for CXL patients

In [ ]:
df_penta_wt_cxl = df_penta.merge(df_cxl, how='inner',on='id')

In [ ]:
df_penta_wt_cxl.to_excel('preprocess_data/Full_Pentacam_for_CXL_patients_only.xlsx')
df_penta_wt_cxl.drop_duplicates(subset=['id'])[['Last Name:','First Name:','id']].to_excel('preprocess_data/IDs_Only_Pentacam_for_CXL_patients_only.xlsx')

In [ ]:
df_penta_wt_cxl[['Last Name:','First Name:','id','first visit date','eye_id']]

In [ ]:
df_cxl.columns

Pentacam data for NON CXL patients

In [ ]:
df_penta_no_cxl = df_penta[~df_penta.id.isin(df_cxl.id)]

In [ ]:
df_penta_no_cxl.to_excel('preprocess_data/Full_Pentacam_for_NON_CXL_patients_only.xlsx')
df_penta_no_cxl.drop_duplicates(subset=['id'])[['Last Name:','First Name:','id']].to_excel('preprocess_data/IDs_Only_Pentacam_for_NON_CXL_patients_only.xlsx')

In [ ]:
#remove the ones who had Pentacam data but it was already after CXL

### 221017

#### Load Data

In [ ]:
df_cxl_clinic = pd.read_excel('data/post_dor_eye_edit/CXL_Clinic_After_Manual_Edit.xlsx')
df_cxl_clinic.id = df_cxl_clinic.id.astype(str).apply(canonize_id)

df_kc_all = pd.read_excel('data/post_dor_eye_edit/KC_Diagnosis_All.xlsx')
df_kc_all.id = df_kc_all.id.astype(str).apply(canonize_id)

df_penta = pd.read_excel('data/raw/Pentacam_raw_data.xlsx')
df_penta = df_penta.rename(columns={'Pat-ID:':'id'})
df_penta.id = df_penta.id.astype(str).apply(canonize_id)

#### Trusted_with_pentacam

In [ ]:
df_trusted_with_pentacam = df_cxl_clinic[df_cxl_clinic.id.isin(df_penta.id.unique())]
df_trusted_with_pentacam

In [ ]:
df_trusted_with_pentacam.to_csv('data/post_dor_eye_edit/CXL_Clinic_Trusted_With_Pentacam_data.csv',index=False)
df_trusted_with_pentacam.to_excel('data/post_dor_eye_edit/CXL_Clinic_Trusted_With_Pentacam_data.xlsx')

#### IDs_from_pentacam_not_in_cxl_clinic



In [ ]:
df_pentacam_not_cxl_clinic = pd.DataFrame(df_penta[~df_penta.id.isin(df_trusted_with_pentacam.id)].id.unique(),columns=['id'])
df_pentacam_not_cxl_clinic

In [ ]:
df_pentacam_not_cxl_clinic.to_csv('data/post_dor_eye_edit/IDs_from_pentacam_not_in_cxl_clinic.csv',index=False)
df_pentacam_not_cxl_clinic.to_excel('data/post_dor_eye_edit/IDs_from_pentacam_not_in_cxl_clinic.xlsx')

#### Pentacam Not CXL With KC Diagnosis

In [ ]:
df_pentacam_not_cxl_clinic[df_pentacam_not_cxl_clinic.id.isin(df_kc_all.id)].to_csv('data/post_dor_eye_edit/IDs_pentacam_not_CXL_with_KC_diagnosis.csv',index=False)
df_pentacam_not_cxl_clinic[df_pentacam_not_cxl_clinic.id.isin(df_kc_all.id)].to_excel('data/post_dor_eye_edit/IDs_pentacam_not_CXL_with_KC_diagnosis.xlsx')

In [ ]:
#now the rest
df_pentacam_not_cxl_clinic[~df_pentacam_not_cxl_clinic.id.isin(df_kc_all.id)].to_csv('data/post_dor_eye_edit/IDs_pentacam_not_CXL_no_KC_diagnosis.csv',index=False)
df_pentacam_not_cxl_clinic[~df_pentacam_not_cxl_clinic.id.isin(df_kc_all.id)].to_excel('data/post_dor_eye_edit/IDs_pentacam_not_CXL_no_KC_diagnosis.xlsx')

### 221204 
Patients after treatment classification_combined CXL and KC diagnosis files


In [6]:
df_penta = pd.read_excel('data/raw/Pentacam_raw_data.xlsx')
df_penta = df_penta.rename(columns={'Pat-ID:':'id'})
df_penta.id = df_penta.id.astype(str).apply(canonize_id)

In [4]:
print(f'# Scans: {len(df_penta)}')
print(f'# Patients: {len(df_penta.id.unique())}')



# Scans: 11760
# Patients: 2625


In [4]:
#Excel Version
# df_combined_KC = pd.read_excel('data/221204_v2/Combined_excel_KC.xlsx').drop(columns=['Unnamed: 0'])
# df_combined_KC = df_combined_KC.rename(columns={'ID':'id'})
# df_combined_KC.id = df_combined_KC.id.astype(str).apply(canonize_id)
# for c in ['CXL L','CXL R','KERATOPLASTY L','KERATOPLASTY R']:
#     df_combined_KC[c].fillna('01/01/1000',inplace=True)

#Clean CSV Version
df_combined_KC = pd.read_csv('data/221204_v2/Combined_excel_KC_formatted.csv',dtype={'id':str})
df_combined_KC.id = df_combined_KC.id.astype(str).apply(canonize_id)

#### Standardize Dataframes

##### Pentacam

In [5]:
df_penta['Exam Eye:'] = df_penta['Exam Eye:'].str.rstrip()

In [6]:
df_penta = df_penta.dropna(subset=['Exam Eye:'])

In [7]:
errors, errors_idx = [], []
for idx, row in df_penta.iterrows():
    if type(row['Exam Date:'])==str:
        try:
            #Pentacam date format is american style (M/D/YYYY)
            df_penta.loc[idx,'Exam Date:'] = datetime.strptime(row['Exam Date:'],'%m/%d/%Y')
        except:
            errors.append(row['Exam Date:'])
            errors_idx.append(idx)
    elif type(row['Exam Date:']) == time:
        errors.append(row['Exam Date:'])
        errors_idx.append(idx)
            
errors

[datetime.time(13, 22, 23),
 datetime.time(14, 16, 29),
 datetime.time(9, 42, 54),
 datetime.time(12, 0, 4),
 datetime.time(12, 1, 44),
 datetime.time(12, 2, 27),
 datetime.time(12, 50, 38),
 datetime.time(12, 51, 19),
 datetime.time(13, 8, 1),
 datetime.time(13, 9, 33),
 datetime.time(15, 19, 40),
 datetime.time(15, 20, 55),
 datetime.time(14, 1, 38),
 datetime.time(14, 2, 35)]

In [8]:
df_penta = df_penta.drop(errors_idx,axis=0)

##### CXL Patients

In [9]:
for col in ['CXL R','CXL L','KERATOPLASTY R','KERATOPLASTY L']:
    df_combined_KC[col] = df_combined_KC[col].apply(lambda x: datetime.strptime(x.lstrip().rstrip(),'%d/%m/%Y'))

In [10]:
df_combined_KC

,id,KC diagnosis,CXL R,CXL L,KERATOPLASTY R,KERATOPLASTY L,NOTES
0,20633426,B,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN
1,30844819,B,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN
2,02863126,B,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN
3,31863842,B,2012-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN
4,05337945,B,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,2009-06-30 00:00:00,NaN
...,...,...,...,...,...,...,...
753,06649437,NaN,2015-10-27 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN
754,06650659,NaN,3000-01-01 00:00:00,2016-08-16 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN
755,06669326,NaN,3000-01-01 00:00:00,2017-09-06 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN
756,06674705,NaN,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,3000-01-01 00:00:00,NaN


#### Split Pentacam By Dates

In [11]:
lbl0, lbl1 = [],[]

##TODO: check for earliest date between CXL & Kerta
na_date = datetime.strptime('01/01/3000','%d/%m/%Y')
for idx, row in df_combined_KC.iterrows():
    if row['CXL R'] == na_date and row['KERATOPLASTY R'] == na_date:
        lbl0.append(df_penta[(df_penta.id == row['id']) & (df_penta['Exam Eye:']=='Right')])
    if row['CXL L'] == na_date and row['KERATOPLASTY L'] == na_date:
        lbl0.append(df_penta[(df_penta.id == row['id']) & (df_penta['Exam Eye:']=='Left')])

    if row['CXL R'] != na_date:
        lbl1.append(df_penta[(df_penta.id==row['id']) & (df_penta['Exam Eye:']=='Right') & (df_penta['Exam Date:'] <= row['CXL R']) ])
    if row['KERATOPLASTY R']!= na_date:
        lbl1.append(df_penta[(df_penta.id==row['id']) & (df_penta['Exam Eye:']=='Right') & (df_penta['Exam Date:'] <= row['KERATOPLASTY R']) ])
    if row['CXL L']!= na_date:
        lbl1.append(df_penta[(df_penta.id==row['id']) & (df_penta['Exam Eye:']=='Left') & (df_penta['Exam Date:'] <= row['CXL L']) ])
    if row['KERATOPLASTY L']!= na_date:
        lbl1.append(df_penta[(df_penta.id==row['id']) & (df_penta['Exam Eye:']=='Left') & (df_penta['Exam Date:'] <=row['KERATOPLASTY L']) ])

In [12]:
df_l1 = pd.concat(lbl1, ignore_index=True)
df_l0 = pd.concat(lbl0, ignore_index=True)

In [13]:
df_l0 = df_l0.drop_duplicates()
df_l1 = df_l1.drop_duplicates()
df_l0['y'] = 0
df_l1['y'] = 1
df = pd.concat([df_l0,df_l1],ignore_index=True)

In [14]:
df

,Last Name:,First Name:,id,D.o.Birth:,Exam Date:,Exam Time:,Exam Eye:,Exam Type:,Exam Comment:,Status,...,D6mm Prog,D8mm Prog,D10mm Prog,KMax Sagittal Front (D),Measuring Head Status:,Measuring Head Status Text:,Unnamed: 74,Unnamed: 75,Unnamed: 76,y
0,ABU MULAHEM,AHMED,20633426,1995-01-01 00:00:00,2017-03-09 00:00:00,10:53:16,Right,(25) 3D-Scan HR,NaN,OK,...,21.0,37.0,56.0,50.8,2,OK,NaN,NaN,NaN,0
1,ABU MULAHEM,AHMED,20633426,1995-01-01 00:00:00,2017-03-09 00:00:00,10:55:20,Left,(25) 3D-Scan HR,NaN,OK,...,31.0,49.0,0.0,60.2,2,OK,NaN,NaN,NaN,0
2,Taha,Salach,30844819,1993-01-01 00:00:00,2017-03-09 00:00:00,12:43:45,Right,(25) 3D-Scan HR,NaN,Lost Segments (Blinking) !,...,23.0,38.0,0.0,47.2,2,OK,NaN,NaN,NaN,0
3,Taha,Salach,30844819,1993-01-01 00:00:00,2017-03-09 00:00:00,12:44:22,Right,(25) 3D-Scan HR,NaN,OK,...,21.0,35.0,58.0,47.2,2,OK,NaN,NaN,NaN,0
4,Taha,Salach,30844819,1993-01-01 00:00:00,2018-03-18 00:00:00,09:34:19,Right,(25) 3D-Scan HR,NaN,OK,...,21.0,35.0,59.0,47.5,2,OK,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2823,Issa,Shorok,06498680,1982-01-01 00:00:00,2018-02-21 00:00:00,11:40:44,Left,(25) 3D-Scan HR,NaN,OK,...,22.0,38.0,0.0,46.0,2,OK,NaN,NaN,NaN,1
2824,Issa,Shorok,06498680,1982-01-01 00:00:00,2018-04-07 00:00:00,08:28:11,Left,(25) 3D-Scan HR,NaN,Alignment (XY) Error,...,18.0,32.0,0.0,47.6,2,OK (Measuring head status),HR (70900),NaN,NaN,1
2825,GRINSHPUN,ODED,06602257,1982-01-09 00:00:00,2020-01-22 00:00:00,09:42:38,Left,(25) 3D-Scan HR,NaN,OK,...,14.0,22.0,34.0,43.3,2,OK (Measuring head status),HR (70900),NaN,NaN,1
2826,REICHENBERG,NETANEL,06626107,11/13/1982,2021-06-21 00:00:00,08:24:57,Left,(25) 3D-Scan HR,NaN,OK,...,64.0,0.0,0.0,53.2,2,OK (Measuring head status),HR (70900),NaN,NaN,1


In [20]:
df['Exam Date:'] = df['Exam Date:'].apply(pd.to_datetime)
df['D.o.Birth:'] = df['D.o.Birth:'].apply(pd.to_datetime)

In [22]:
df['age'] = df.apply(lambda row: row['Exam Date:'].year - row['D.o.Birth:'].year, axis=1)

In [23]:
df.to_csv('data/221204_v2/KC_filtered_by_date_labeled.csv', index=False)

## Masking and Joining

In [37]:
df_cxl = pd.read_excel('data/221204_v2/Combined_excel_KC_formatted.xlsx')
df_cxl['ID'] = df_cxl['ID'].astype(str).apply(canonize_id)
df_cxl = df_cxl.rename(columns={'ID':'idp'})
df_cxl['idp'] = df_cxl['idp'].astype(str)


df_penta = pd.read_excel('data/raw/Pentacam_raw_data.xlsx')
df_penta = df_penta.rename(columns={'Pat-ID:':'idp'})
df_penta.idp = df_penta.idp.astype(str).apply(canonize_id)
df_penta['idp'] = df_penta['idp'].astype(str)


In [45]:
df_m = pd.merge(df_penta,df_cxl,'left',on='idp')
ids_random, double_ret = {},[]
def get_rand_id(idp):
    if idp not in ids_random:
        randval = str(np.random.randint(10000,400000))
        while randval in double_ret:
            randval = str(np.random.randint(10000,400000))
        double_ret.append(randval)
        ids_random[idp] = randval
    return ids_random[idp]
df_m['index'] = df_m.idp.apply(get_rand_id)


In [46]:
df_m['index']

0        149980
1        149980
2        148354
3        148354
4        148354
          ...  
11755    232188
11756    250310
11757    250310
11758    185930
11759    185930
Name: index, Length: 11760, dtype: object

In [48]:
df_m.to_excel('data/221204_v2/Raw_Combined_IdsMesh.xlsx',index=False)
df_m[['idp','index']].to_excel('data/221204_v2/Id_Index_Mapping.xlsx',index=False)